In [4]:
import os
import re
import psutil
from tqdm import tqdm
from collections import defaultdict
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd

from transformers import AutoTokenizer, AutoModel, AutoModelForTokenClassification
from transformers import pipeline
from sentence_transformers import SentenceTransformer

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.feature_extraction.text import TfidfVectorizer

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

pool = ThreadPoolExecutor()
default_workers_threads = pool._max_workers

print(f"CPU count: {os.cpu_count()}")
print(f"Memory GB: {psutil.virtual_memory().total >> 30}")
print(f"Default thread workers: {default_workers_threads}")

CPU count: 16
Memory GB: 31
Default thread workers: 20


In [5]:
def explode_subjects(x):
    pattern = r'([^();]+) \(([\w.]+)\)'
    matches = re.findall(pattern, x)

    subject_list = []
    key_list = []

    for subject, key in matches:
        subject_list.append(subject.strip())
        key_list.append(key.strip())

    return subject_list, key_list

In [6]:
df = pd.read_parquet("arxiv_audio+recognition_search_term.parquet")
df.head()

,title,authors,abstract,Journal,code,submitted_date,orginally_announced_date,subjects
0,M&M: Multimodal-Multitask Model Integrating Au...,"Long Nguyen-Phuoc,Renald Gaboriau,Dimitri Dela...","This paper introduces the M&M model, a novel m...",Proceedings of the 19th International Joint Co...,arXiv:2403.09451,2024-03-14,2024-03-01,Computation and Language (cs.CL); Sound (cs.SD...
1,Unsupervised Modality-Transferable Video Highl...,"Tingtian Li,Zixun Sun,Xinyu Xiao",Identifying highlight moments of raw video mat...,None,arXiv:2403.09401,2024-03-14,2024-03-01,Sound (cs.SD); Computation and Language (cs.CL...
2,More than words: Advancements and challenges i...,Anna Kruspe,This paper addresses the challenges and advanc...,None,arXiv:2403.09298,2024-03-14,2024-03-01,Audio and Speech Processing (eess.AS); Computa...
3,SpeechColab Leaderboard: An Open-Source Platfo...,"Jiayu Du,Jinpeng Li,Guoguo Chen,Wei-Qiang Zhang",In the wake of the surging tide of deep learni...,None,arXiv:2403.08196,2024-03-12,2024-03-01,Audio and Speech Processing (eess.AS); Computa...
4,Automatic Speech Recognition (ASR) for the Dia...,"Taekyung Ahn,Yeonjung Hong,Younggon Im,Do Hyun...",This study presents a model of automatic speec...,None,arXiv:2403.08187,2024-03-12,2024-03-01,Computer Vision and Pattern Recognition (cs.CV...


In [7]:
all_subjects = []
all_keys = []

for val in tqdm(df['subjects'].values):
    subject_list, key_list = explode_subjects(val)

    all_subjects.extend(subject_list)
    all_keys.extend(key_list)

100%|██████████| 4000/4000 [00:00<00:00, 240099.83it/s]


In [8]:
abstracts = df['abstract'].tolist()

# Abstracts embeddings

In [9]:
# Load the pre-trained model and tokenizer
model_name = "allenai/scibert_scivocab_uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

In [10]:
nltk.download('punkt')
nltk.download('stopwords')

stop_words = stopwords.words('english')
stemmer = PorterStemmer()

def preprocess_text(text):
  text = text.lower()
  text = ''.join([char for char in text if char.isalnum() or char in ' '])
  words = [word for word in text.split() if word not in stop_words]
  stemmed_words = [stemmer.stem(word) for word in words]
  return ' '.join(stemmed_words)

#preprocessed_abstracts = [preprocess_text(text) for text in abstracts]

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\517\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\517\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [11]:
# Generate encoded abstracts
#encoded_abstracts = tokenizer(abstracts, padding = True)
#encoded_preprocessed_abstracts = tokenizer(preprocessed_abstracts, padding = True)

# Abstracts NER

In [12]:
tokenizer = AutoTokenizer.from_pretrained("RJuro/SciNERTopic")
model_trf = AutoModelForTokenClassification.from_pretrained("RJuro/SciNERTopic")

nlp = pipeline("ner", model=model_trf, tokenizer=tokenizer, aggregation_strategy = 'average')

In [13]:
result_dict = defaultdict(list)

def extract_ner_worker(abstract, lock):
    local_result_dict = defaultdict(list)
    result = nlp(abstract)
    for item in result:
        entity_group = item['entity_group']
        word = item['word']
        local_result_dict[entity_group].append(word)
    
    with lock:
        for key, value in local_result_dict.items():
            result_dict[key].extend(value)

In [16]:
lock = threading.Lock()  # Create a lock for synchronizing access to result_dict

with ThreadPoolExecutor(max_workers = 12) as executor, tqdm(total=len(abstracts)) as pbar:
    futures = [executor.submit(extract_ner_worker, abstract, lock) for abstract in abstracts]
    for future in as_completed(futures):
        pbar.update()

  0%|          | 0/4000 [00:00<?, ?it/s]Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length.

In [17]:
import pickle

# Specify the file path where you want to save the pickle file
file_path = "paper_abstracts_ner_dict.pkl"

# Save result_dict to a pickle file
with open(file_path, "wb") as file:
    pickle.dump(result_dict, file)